In [ ]:
class ItemTienda:
    # Atributo de clase autoincremental: cada ítem recibe un código único
    _contador_codigo = 1

    def __init__(self, nombre, precio, nivel_requerido, descripcion):
        # Se valida antes de asignar el código para no "gastar" códigos en ítems inválidos
        if precio < 0:
            raise ValueError("El precio no puede ser negativo.")
        if nivel_requerido < 1:
            raise ValueError("El nivel requerido debe ser al menos 1.")
        self.codigo = ItemTienda._contador_codigo
        ItemTienda._contador_codigo += 1
        self.nombre = nombre
        self.precio = precio
        self.nivel_requerido = nivel_requerido
        self.descripcion = descripcion

    # Aplica el efecto del ítem al jugador. Devuelve True si se aplicó (y por tanto se cobra)
    def aplicar(self, jugador):
        raise NotImplementedError("Cada tipo de ítem define su propio efecto.")


class Equipo(ItemTienda):
    def __init__(self, nombre, precio, nivel_requerido, descripcion, bonificacion):
        if bonificacion <= 0:
            raise ValueError("La bonificación debe ser positiva.")
        super().__init__(nombre, precio, nivel_requerido, descripcion)
        self.bonificacion = bonificacion    # puntos que aumentan la hp_maxima del jugador

    def aplicar(self, jugador):
        # Se guarda en el inventario y mejora la vida máxima (sin repetir el mismo equipo)
        if self in jugador.inventario:
            print(f"Ya tienes equipado: {self.nombre}.")
            return False
        jugador.inventario.append(self)
        jugador.hp_maxima += self.bonificacion
        print(f"Has equipado: {self.nombre}. Bonificación: +{self.bonificacion} HP máxima")
        return True


class Pocion(ItemTienda):
    def __init__(self, nombre, precio, nivel_requerido, descripcion, hp_restaurada):
        if hp_restaurada <= 0:
            raise ValueError("La vida restaurada debe ser positiva.")
        super().__init__(nombre, precio, nivel_requerido, descripcion)
        self.hp_restaurada = hp_restaurada

    def aplicar(self, jugador):
        # Cura al jugador respetando su hp_maxima (usa Jugador.curar)
        if jugador.hp >= jugador.hp_maxima:
            print("Tu salud ya está al máximo; no se usó la poción.")
            return False
        hp_antes = jugador.hp
        jugador.curar(self.hp_restaurada)
        print(f"Te has curado {jugador.hp - hp_antes} HP. Salud actual: {jugador.hp}/{jugador.hp_maxima}")
        return True


class Recompensa(ItemTienda):
    def __init__(self, nombre, precio, nivel_requerido, descripcion, creada_por_usuario):
        super().__init__(nombre, precio, nivel_requerido, descripcion)
        self.creada_por_usuario = creada_por_usuario

    def aplicar(self, jugador):
        # Solo se registra el premio real
        print(f"Recompensa canjeada: {self.nombre}. ¡Disfruta tu premio!")
        return True


class Tienda:
    def __init__(self):
        self.items = []    # Lista de objetos ItemTienda

    def agregar_item(self, item):
        if not isinstance(item, ItemTienda):
            raise TypeError("Solo se pueden agregar objetos de tipo ItemTienda.")
        if item not in self.items:
            self.items.append(item)

    def mostrar_tabla(self):
        print("\n--- OBJETOS DE LA TIENDA ---")
        if not self.items:
            print("La tienda está vacía.")
            return
        for item in self.items:
            print(f"[{item.codigo}] {item.nombre} | Nivel req: {item.nivel_requerido} | Precio: {item.precio} oro")
            print(f"    Desc: {item.descripcion}")

    def buscar_por_codigo(self, codigo):
        for item in self.items:
            if item.codigo == codigo:
                return item
        return None

    # Compra un ítem: valida existencia, nivel y oro; cobra solo si el ítem se aplicó
    def comprar(self, jugador, codigo):
        item = self.buscar_por_codigo(codigo)
        if item is None:
            print(f"No existe un ítem con código {codigo}.")
            return False
        if jugador.nivel < item.nivel_requerido:
            print(f"Necesitas nivel {item.nivel_requerido} para comprar {item.nombre}.")
            return False
        if jugador.oro < item.precio:
            print(f"Oro insuficiente para {item.nombre}: tienes {jugador.oro} y cuesta {item.precio}.")
            return False
        if not item.aplicar(jugador):
            return False
        jugador.oro -= item.precio
        print(f"Compra realizada: {item.nombre} (-{item.precio} oro). Oro restante: {jugador.oro}")
        return True